In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from joblib import Parallel, delayed

import warnings
warnings.filterwarnings('ignore')

ModuleNotFoundError: No module named 'numpy'

In [ ]:

TARGET_COL = 'RIDAGEYR'

FEATURE_COLS_BLOOD_PRESSURE = ['BPXSY1', 'BPXDI1', 'BPXPLS']
FEATURE_COLS_METABOLIC      = ['LBXGH', 'BMXBMI', 'BMXWAIST']
FEATURE_COLS_BODY           = ['BMXLEG', 'BMXARML', 'BMXARMC']
FEATURE_COLS_LIFESTYLE      = ['PAD680', 'SMQ020']
FEATURE_COLS_REPORTED       = ['BPQ080', 'BPQ020']

ALL_FEATURE_COLS = (
    FEATURE_COLS_BLOOD_PRESSURE +
    FEATURE_COLS_METABOLIC +
    FEATURE_COLS_BODY +
    FEATURE_COLS_LIFESTYLE +
    FEATURE_COLS_REPORTED
)

DEFAULT_CONFIG = {
    'learning_rate': 0.1,
    'max_depth': 6,
    'n_estimators': 300,
    'colsample_bytree': 0.75,
    'subsample': 0.8,
}

PARAM_GRID = {
    'n_estimators':   [100, 300, 500],
    'max_depth':      [4, 6, 8],
}

TEST_SIZE   = 0.2
RANDOM_SEED = 42
MIN_AGE     = 18  

In [ ]:
def load_data(path, min_age=MIN_AGE):
    df = pd.read_csv(path)
    df = df[df[TARGET_COL] >= min_age].reset_index(drop=True)
    print(f'Age distribution:')
    print(df[TARGET_COL].describe().round(2))
    return df


def split_data(df):
    X = np.concatenate([
        df[FEATURE_COLS_BLOOD_PRESSURE].values,
        df[FEATURE_COLS_METABOLIC].values,
        df[FEATURE_COLS_BODY].values,
        df[FEATURE_COLS_LIFESTYLE].values,
        df[FEATURE_COLS_REPORTED].values,
    ], axis=1)

    y = df[TARGET_COL].values
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED
    )
    print(f'Train: {X_train.shape[0]} | Test: {X_test.shape[0]}')
    return X_train, X_test, y_train, y_test

df = load_data('nhanes_clean.csv')
X_train, X_test, y_train, y_test = split_data(df)

In [ ]:
class Model:
    def __init__(self, config=DEFAULT_CONFIG, n_jobs=2):
        self.model = XGBRegressor(
            **config,
            objective='reg:squarederror',
            random_state=RANDOM_SEED,
            n_jobs=n_jobs,
        )
        self.is_fitted = False
        self.config = config


def train_model(model, X_train, y_train):
    model.model.fit(X_train, y_train)
    model.is_fitted = True

    train_loss = loss(model, X_train, y_train)
    print(f'Train error: {np.sqrt(train_loss):.4f} years')
    return model


def pred(model, X):
    if not model.is_fitted:
        raise RuntimeError('untrained model')
    return model.model.predict(X)


def loss(model, X, y):
    preds = model.model.predict(X)
    return np.mean((preds - y) ** 2)  


def evaluate(model, X_test, y_test, label='XGBoost'):
    preds = pred(model, X_test)
    rmse  = np.sqrt(mean_squared_error(y_test, preds))
    mae   = mean_absolute_error(y_test, preds)
    corr  = np.corrcoef(y_test, preds)[0, 1]
    print(f'{label}')
    print(f'RMSE: {rmse:.4f} years')
    print(f'MAE:  {mae:.4f} years')
    print(f'Pearson r: {corr:.4f}')
    return preds, rmse, mae

In [ ]:
xgb_model = Model(config=DEFAULT_CONFIG)
xgb_model = train_model(xgb_model, X_train, y_train)
xgb_preds, xgb_rmse, xgb_mae = evaluate(xgb_model, X_test, y_test, label='XGBoost')


In [ ]:
def grid_search(X_train, y_train, X_test, y_test, n_jobs=2, top_n=5):
    def train_eval(config):
        model = Model(config=config)
        model.model.fit(X_train, y_train)
        model.is_fitted = True
        val_loss = loss(model, X_test, y_test)
        return {'model': model, 'config': config, 'val_loss': val_loss}

    combos = list(ParameterGrid(PARAM_GRID))
    print(f'Testing {len(combos)} configurations...')

    results = Parallel(n_jobs=n_jobs, verbose=5)(
        delayed(train_eval)(config) for config in combos
    )

    sorted_res = sorted(results, key=lambda x: x['val_loss'])[:top_n]

    print(f'Top {top_n} configurations:')
    for i, r in enumerate(sorted_res):
        print(f'  Rank {i+1} | RMSE: {np.sqrt(r["val_loss"]):.4f} | Config: {r["config"]}')

    return sorted_res


top = grid_search(X_train, y_train, X_test, y_test)
best = top[0]['model']

In [ ]:
best_preds, best_rmse, best_mae = evaluate(best, X_test, y_test, label='XGBoost (best config)')


bio_age_gap = best_preds - y_test
print(f'Biological age difference (predicted - actual):')
print(f'  Mean:   {bio_age_gap.mean():.4f} years')
print(f'  Std:    {bio_age_gap.std():.4f} years')
print(f'  Median: {np.median(bio_age_gap):.4f} years')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.scatter(y_test, best_preds, alpha=0.3, s=10, color='steelblue')
ax.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5, label='Perfect prediction')
ax.set_xlabel('Actual Age (years)')
ax.set_ylabel('Predicted Age (years)')
ax.set_title('Predicted vs Actual Age')
ax.legend()

ax = axes[1]
ax.hist(bio_age_gap, bins=50, color='steelblue', edgecolor='white', linewidth=0.5)
ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='Zero gap')
ax.axvline(bio_age_gap.mean(), color='orange', linestyle='--', linewidth=1.5, label=f'Mean = {bio_age_gap.mean():.2f}')
ax.set_xlabel('Biological Age Gap (years)')
ax.set_ylabel('Count')
ax.set_title('Biological Age Gap Distribution')
ax.legend()

ax = axes[2]
importances = best.model.feature_importances_
indices     = np.argsort(importances)[::-1]
sorted_features = [ALL_FEATURE_COLS[i] for i in indices]
sorted_importances = importances[indices]

ax.barh(sorted_features[::-1], sorted_importances[::-1], color='steelblue')
ax.set_xlabel('Feature Importance (F-score)')
ax.set_title('XGBoost Feature Importance')

plt.tight_layout()
plt.savefig('nhanes_results.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:

_, df_test = train_test_split(df[df[TARGET_COL] >= MIN_AGE].reset_index(drop=True),
                               test_size=TEST_SIZE, random_state=RANDOM_SEED)
df_test = df_test.copy()
df_test['predicted_age'] = best_preds
df_test['bio_age_gap']   = bio_age_gap

def ablation_report(df, split_col, split_labels, title):
    print(f'Ablation: {title}')
    for val, label in split_labels.items():
        subset = df[df[split_col] == val]
        gap    = subset['bio_age_gap']
        rmse   = np.sqrt(mean_squared_error(subset[TARGET_COL], subset['predicted_age']))
        print(f'  {label:20s} | n={len(subset):5d} | Mean gap: {gap.mean():+.2f} yrs | RMSE: {rmse:.2f} yrs')

ablation_report(df_test, 'SMQ020', {0: 'Never smoked', 1: 'Ever smoked'}, 'Smoking')

ablation_report(df_test, 'BPQ020', {0: 'Normal BP', 1: 'High BP'}, 'Blood Pressure')

ablation_report(df_test, 'BPQ080', {0: 'Normal cholesterol', 1: 'High cholesterol'}, 'Cholesterol')

median_sedentary = df_test['PAD680'].median()
df_test['is_sedentary'] = (df_test['PAD680'] >= median_sedentary).astype(int)
ablation_report(df_test, 'is_sedentary', {0: 'Active', 1: 'Sedentary'}, 'Sedentary Time')

In [ ]:

print(f'XGBoost (default)RMSE: {xgb_rmse:.4f} | MAE: {xgb_mae:.4f}')
print(f'XGBoost (tuned) RMSE: {best_rmse:.4f} | MAE: {best_mae:.4f}')
print(f'Best config: {top[0]["config"]}') 
print(f' Biological age gap mean: {bio_age_gap.mean():+.2f} yrs, std: {bio_age_gap.std():.2f} yrs')